<a href="https://colab.research.google.com/github/lricci03/Hands-on-ML/blob/main/c10_ex14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Solution to Chapter 10 - ex 14.

Create a custom Dense module that replicates the functionality of an nn.Linear module followed by an nn.ReLU module.

Try implementing it first using the nn.Linear and nn.ReLU modules, and then reimplement it using nn.Paramter and the relu() function

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class Dense(nn.Module): # inherit from nn.Module
  def __init__(self,n_inputs,n_outputs):
    super().__init__()  # initialize the parent class nn.Module
    self.n_inputs = n_inputs    # variable that gives the no of inputs of the nn
    self.n_outputs = n_outputs  # variable that gives the no of outputs of the nn
    # the stack needs to be created outside the forward() function:
    # it is created when we initialize an instance of the class, so weights are assigned only once when we initialize the instance
    # if we define it inside the forward() function: it is created everytime we call the function
    # thus new weights are assigned everytime we call the function.
    self.stack = nn.Sequential(
        nn.Linear(self.n_inputs, self.n_outputs), nn.ReLU()
    )

  def forward(self,X):
    return self.stack(X)

Try out the class

In [ ]:
torch.manual_seed(42) # to get reproducible results

In [ ]:
dense = Dense(3,1)
X = torch.randn(5,3) # 5 instances with 3 features each -> 5x3 tensor
X

tensor([[ 0.2345,  0.2303, -1.1229],
        [-0.1863,  2.2082, -0.6380],
        [ 0.4617,  0.2674,  0.5349],
        [ 0.8094,  1.1103, -1.6898],
        [-0.9890,  0.9580,  1.3221]])

In [ ]:
dense.n_inputs, dense.n_outputs

(3, 1)

Make a prediction

In [ ]:
y_pred = dense(X)
y_pred

tensor([[0.8961],
        [1.5926],
        [0.7899],
        [1.6482],
        [0.3741]], grad_fn=<ReluBackward0>)

Check that it is correct

In [ ]:
# The NN model is given by the attribute dense.stack (we defined it self.stack = nn.Sequential...)
# The layers can be accessed as if dense.stack was a list: dense.stack[0], dense.stack[1], ...
weights = dense.stack[0].weight
bias = dense.stack[0].bias
print('Weights are: ', weights, 'bias is: ', bias)

Weights are:  Parameter containing:
tensor([[ 0.4414,  0.4792, -0.1353]], requires_grad=True) bias is:  Parameter containing:
tensor([0.5304], requires_grad=True)


In [ ]:
print('weights shape: ', weights.shape, 'X shape: ', X.shape, 'bias shape: ', bias.shape)

weights shape:  torch.Size([1, 3]) X shape:  torch.Size([5, 3]) bias shape:  torch.Size([1])


In [ ]:
X @ weights.T + bias


tensor([[0.8961],
        [1.5926],
        [0.7899],
        [1.6482],
        [0.3741]], grad_fn=<AddBackward0>)

In [ ]:
torch.relu(X@weights.T+bias) == y_pred

tensor([[True],
        [True],
        [True],
        [True],
        [True]])

## NN with two neurons and 3 features

In [ ]:
model = Dense(3,2)
model.stack[0].weight

Parameter containing:
tensor([[ 0.3337, -0.2524,  0.3333],
        [ 0.1033,  0.2932, -0.3519]], requires_grad=True)

## Implementation using nn.Parameter and relu() function

In [ ]:
x = nn.Parameter(torch.randn(2,3),requires_grad=False)
x

Parameter containing:
tensor([[ 0.1331,  0.8640, -1.0157],
        [-0.8887,  0.1498, -0.2089]])

In [ ]:
class Dense2(nn.Module):
  def __init__(self, n_inputs, n_outputs):
    super().__init__()
    self.n_inputs = n_inputs
    self.n_outputs = n_outputs
    # define the weights parameters and the bias
    self.weights = nn.Parameter(torch.randn(n_outputs,n_inputs), requires_grad=True) # dim = 1 x n_inputs
    self.bias = nn.Parameter(torch.randn(n_outputs),requires_grad=True) # dim = 1

  def forward(self,X):
    linear = X @ self.weights.T + self.bias
    return torch.relu(linear)

In [ ]:
dense2 = Dense2(3,2) # 3 features, 2 neurons
Y = torch.randn(5,3)

In [ ]:
Y_pred = dense2(Y)
Y_pred

tensor([[2.0311, 0.0000],
        [1.5611, 0.0000],
        [0.5513, 0.0000],
        [2.4661, 0.0403],
        [2.3215, 0.8919]], grad_fn=<ReluBackward0>)

In [ ]:
dense2.weights

Parameter containing:
tensor([[-0.3870,  0.9912,  0.4679],
        [-0.2049, -0.7409,  0.3618]], requires_grad=True)